In [1]:
import lfox
import lfox.lattice as lat
import lfox.evolution.hmc as lhmc
import jax
import jax.numpy as jnp
import numpy as np

import lfox.fermions.spin as spin

# Imports below require "dev" environment
import matplotlib.pyplot as plt
import lsqfit
import gvar as gv
import tqdm

# Double precision!
jax.config.update("jax_enable_x64", True)
jax.config.update("jax_threefry_partitionable", True)

In [2]:
Lat4 = lat.SquareLattice(dims=(4,4,8))
t1 = lat.LatticeField(Lat4, indices=(4,))
print(t1.dims)
t1.F = t1.F.at[0,0,0].set(1)
print(t1.F[0,0,0])

I0000 00:00:1696822388.309241       1 tfrt_cpu_pjrt_client.cc:349] TfrtCpuClient created.


(4, 4, 8, 4)
[1. 1. 1. 1.]


In [3]:
Pauli_X = np.array([[0,1],[1,0]])
Pauli_Y = np.array([[0,-1j],[1j,0]])
Pauli_Z = np.array([[1,0],[0,-1]])

In [4]:
# DeGrand-Rossi gamma basis

Gamma_0 = np.array([
    [0, 0, 0, 1j],
    [0, 0, 1j, 0],
    [0, -1j, 0, 0],
    [-1j, 0, 0, 0],
])
Gamma_1 = np.array([
    [0, 0, 0, -1],
    [0, 0, 1, 0],
    [0, 1, 0, 0],
    [-1, 0, 0, 0],
])
Gamma_2 = np.array([
    [0, 0, 1j, 0],
    [0, 0, 0, -1j],
    [-1j, 0, 0, 0],
    [0, 1j, 0, 0],
])
Gamma_3 = np.array([
    [0, 0, 1, 0],
    [0, 0, 0, 1],
    [1, 0, 0, 0],
    [0, 1, 0, 0],
])
Gamma_5 = np.array([
    [1, 0, 0, 0],
    [0, 1, 0, 0],
    [0, 0, -1, 0],
    [0, 0, 0, -1],
])

In [5]:
# Euclidean product of all four gammas --> gamma_5
np.all(Gamma_0 @ Gamma_1 @ Gamma_2 @ Gamma_3 == Gamma_5)

True

In [6]:
Lat4 = lat.SquareLattice(dims=(2,3))
psi = spin.Dirac4DFermionField(Lat4).unit_fill()
chi = spin.Dirac4DFermionField(Lat4).unit_fill()


In [7]:
(psi.conj() * chi).F

Array([[[1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.]],

       [[1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.]]], dtype=float64)

In [8]:
M = np.array([[1,2],[3,4]])
np.einsum('...i',M)

array([[1, 2],
       [3, 4]])

In [9]:
def spin_gamma_product(psi_L, Gamma, psi_R):
    return jnp.einsum('i,ij,j', psi_L.conj(), Gamma, psi_R)

@jax.jit
def gp2(psi_L, Gamma, psi_R):
    return jnp.einsum('...i,ij,...j', psi_L.conj(), Gamma, psi_R)

In [10]:
print(spin_gamma_product(np.ones(4), spin.GammaMatrix[5], np.ones(4)))

0.0


In [11]:
c = (3,3)
c + (4,5)

(3, 3, 4, 5)

In [12]:
X = jnp.tensordot(spin.GammaMatrix['I'], chi.F, ([0],[2]))
X, X.shape, chi.F.shape

(Array([[[1., 1., 1.],
         [1., 1., 1.]],
 
        [[1., 1., 1.],
         [1., 1., 1.]],
 
        [[1., 1., 1.],
         [1., 1., 1.]],
 
        [[1., 1., 1.],
         [1., 1., 1.]]], dtype=float64),
 (4, 2, 3),
 (2, 3, 4))

In [13]:
print(gp2(psi.F, spin.GammaMatrix['I'], chi.F))

td = jnp.tensordot(psi.F, chi.F, ([2],[2]))
print(td)
print(td.shape, psi.F.shape)

[[4. 4. 4.]
 [4. 4. 4.]]
[[[[4. 4. 4.]
   [4. 4. 4.]]

  [[4. 4. 4.]
   [4. 4. 4.]]

  [[4. 4. 4.]
   [4. 4. 4.]]]


 [[[4. 4. 4.]
   [4. 4. 4.]]

  [[4. 4. 4.]
   [4. 4. 4.]]

  [[4. 4. 4.]
   [4. 4. 4.]]]]
(2, 3, 2, 3) (2, 3, 4)


In [14]:
vv = lambda x, y: jnp.vdot(x,y)
mv = jax.vmap(vv, (0, None), 0)
mm = jax.vmap(mv, (None, 1), 1)
x = jnp.ones((2,3))
y = jnp.ones((3,4))
z = jnp.ones(3)

print(vv(z,z))
print(mv(x,z))

3.0
[3. 3.]


In [15]:
M = jax.vmap(vv, 0, 0)
M(z,z)

Array([1., 1., 1.], dtype=float64)

In [16]:
lat_sgp = jax.vmap(jax.vmap(spin_gamma_product, (0, None, 0)), (0, None, 0))

In [17]:
%time lat_sgp(psi.F, spin.GammaMatrix[5], chi.F)

CPU times: user 41.2 ms, sys: 2.74 ms, total: 43.9 ms
Wall time: 42.8 ms


Array([[0., 0., 0.],
       [0., 0., 0.]], dtype=float64)

In [18]:
lat_sgp_jit = jax.jit(lat_sgp)

In [19]:
%time lat_sgp_jit(psi.F, spin.GammaMatrix[5], chi.F)

CPU times: user 23.7 ms, sys: 1.86 ms, total: 25.6 ms
Wall time: 24 ms


Array([[0., 0., 0.],
       [0., 0., 0.]], dtype=float64)

In [20]:
%time gp2(psi.F, spin.GammaMatrix[5], chi.F)

CPU times: user 30.8 ms, sys: 2.25 ms, total: 33 ms
Wall time: 31.2 ms


Array([[0., 0., 0.],
       [0., 0., 0.]], dtype=float64)

In [21]:
print(psi.bilinear(chi).F)
print(psi.bilinear(chi, spin_mat=spin.GammaMatrix[5]).F)

[[4. 4. 4.]
 [4. 4. 4.]]
[[0. 0. 0.]
 [0. 0. 0.]]


In [22]:
%time psi.bilinear(chi, spin_mat=spin.GammaMatrix[5])

CPU times: user 493 µs, sys: 361 µs, total: 854 µs
Wall time: 525 µs


In [23]:
psi._tree_flatten()

((Array([[[1., 1., 1., 1.],
          [1., 1., 1., 1.],
          [1., 1., 1., 1.]],
  
         [[1., 1., 1., 1.],
          [1., 1., 1., 1.],
          [1., 1., 1., 1.]]], dtype=float64),),
 {'lattice': <lfox.lattice.SquareLattice at 0x126c88790>,
  'bc': array([1., 1.]),
  'indices': (4,)})

In [24]:
np.indices((2,2,2,2)).sum(axis=0) % 2

array([[[[0, 1],
         [1, 0]],

        [[1, 0],
         [0, 1]]],


       [[[1, 0],
         [0, 1]],

        [[0, 1],
         [1, 0]]]])

In [25]:
def cb(shape):
    return (np.indices(shape).sum(axis=0) % 2) == 0

cb_blk = np.indices((4,4)).sum(axis=0) % 2 == 0
cb_red = ~cb_blk

print(cb_blk, cb_red)
cb_blk.dtype

[[ True False  True False]
 [False  True False  True]
 [ True False  True False]
 [False  True False  True]] [[False  True False  True]
 [ True False  True False]
 [False  True False  True]
 [ True False  True False]]


dtype('bool')

In [26]:
test_shape = (4,4)
Y = np.arange(16).reshape(test_shape)
Z = np.zeros(test_shape)
print(cb(test_shape))
Z[cb(test_shape)] = 2
print(Z)
print(jnp.where(cb(test_shape), Y, 0))

[[ True False  True False]
 [False  True False  True]
 [ True False  True False]
 [False  True False  True]]
[[2. 0. 2. 0.]
 [0. 2. 0. 2.]
 [2. 0. 2. 0.]
 [0. 2. 0. 2.]]
[[ 0  0  2  0]
 [ 0  5  0  7]
 [ 8  0 10  0]
 [ 0 13  0 15]]


In [27]:
Y[cb_blk], Y[cb_red]

(array([ 0,  2,  5,  7,  8, 10, 13, 15]),
 array([ 1,  3,  4,  6,  9, 11, 12, 14]))

In [28]:
sublat = (4,2)
Y_blk = Y[cb_blk].reshape(sublat)
Y_red = Y[cb_red].reshape(sublat)

print(Y_blk, "\n", Y_red)

Y_big_blk = jnp.where(cb_blk, Y, 0)
Y_big_red = jnp.where(cb_red, Y, 0)

print(Y_big_blk)
print(Y_big_red)

[[ 0  2]
 [ 5  7]
 [ 8 10]
 [13 15]] 
 [[ 1  3]
 [ 4  6]
 [ 9 11]
 [12 14]]
[[ 0  0  2  0]
 [ 0  5  0  7]
 [ 8  0 10  0]
 [ 0 13  0 15]]
[[ 0  1  0  3]
 [ 4  0  6  0]
 [ 0  9  0 11]
 [12  0 14  0]]


In [29]:
np.eye(4)

array([[1., 0., 0., 0.],
       [0., 1., 0., 0.],
       [0., 0., 1., 0.],
       [0., 0., 0., 1.]])

In [30]:
## NN, axis 0
print(Y_blk + Y_red)
print(Y_big_blk + np.roll(Y_big_red, axis=1, shift=-1))

[[ 1  5]
 [ 9 13]
 [17 21]
 [25 29]]
[[ 1  0  5  0]
 [ 0 11  0 11]
 [17  0 21  0]
 [ 0 27  0 27]]


In [31]:
cb2 = cb(test_shape).flatten()
cb2
Y[cb(test_shape)]

array([ 0,  2,  5,  7,  8, 10, 13, 15])

In [32]:
Y = np.arange(16).reshape(2,2,4)
cb_test = np.indices((2,2)).sum(axis=0) % 2
print(Y, cb_test)
print(Y[0,1],Y[1,0])

[[[ 0  1  2  3]
  [ 4  5  6  7]]

 [[ 8  9 10 11]
  [12 13 14 15]]] [[0 1]
 [1 0]]
[4 5 6 7] [ 8  9 10 11]


In [33]:
np.broadcast_to(cb_test, (4,2,2)).T
exp_mask = np.repeat(np.expand_dims(cb_test,axis=2), 4, axis=2)

In [34]:
np.where(exp_mask, Y, 0)

array([[[ 0,  0,  0,  0],
        [ 4,  5,  6,  7]],

       [[ 8,  9, 10, 11],
        [ 0,  0,  0,  0]]])

In [35]:
Y

array([[[ 0,  1,  2,  3],
        [ 4,  5,  6,  7]],

       [[ 8,  9, 10, 11],
        [12, 13, 14, 15]]])

In [36]:
Y[::2]

array([[[0, 1, 2, 3],
        [4, 5, 6, 7]]])

In [39]:
psi_shift = WD.shift_fermion(psi, 1, 1)
print(psi_shift.F)
print(spin.GammaMatrix['I'].shape, psi.F.shape)
print(psi._spin_product(spin.GammaMatrix['I'], psi.F))

[[[1. 1. 1. 1.]
  [1. 1. 1. 1.]
  [1. 1. 1. 1.]]

 [[1. 1. 1. 1.]
  [1. 1. 1. 1.]
  [1. 1. 1. 1.]]]
(4, 4) (2, 3, 4)
[[[1. 1. 1. 1.]
  [1. 1. 1. 1.]
  [1. 1. 1. 1.]]

 [[1. 1. 1. 1.]
  [1. 1. 1. 1.]
  [1. 1. 1. 1.]]]


In [42]:
print(psi.F)
from lfox.fermions.wilson import WilsonDiracOp

WD = WilsonDiracOp(kappa=0.18)

print(psi.bc)
print(psi.lattice.d)

chi =  WD.op(psi)
chi.F

[[[1. 1. 1. 1.]
  [1. 1. 1. 1.]
  [1. 1. 1. 1.]]

 [[1. 1. 1. 1.]
  [1. 1. 1. 1.]
  [1. 1. 1. 1.]]]
[1. 1.]
2


Array([[[0.28+0.j, 0.28+0.j, 0.28+0.j, 0.28+0.j],
        [0.28+0.j, 0.28+0.j, 0.28+0.j, 0.28+0.j],
        [0.28+0.j, 0.28+0.j, 0.28+0.j, 0.28+0.j]],

       [[0.28+0.j, 0.28+0.j, 0.28+0.j, 0.28+0.j],
        [0.28+0.j, 0.28+0.j, 0.28+0.j, 0.28+0.j],
        [0.28+0.j, 0.28+0.j, 0.28+0.j, 0.28+0.j]]], dtype=complex128)